# 03 · Evaluation & Demo trên T4


> Môi trường: **Colab → T4 GPU**. Giữ cùng thư mục `PROJECT` qua cả ba notebook.
> Notebook tự chứa mã nguồn; không cần upload thêm ZIP. Chạy từ trên xuống trên một runtime mới.
>
> **Trạng thái bàn giao:** đã kiểm tra mã và tích hợp bằng mô hình nhỏ trên CPU;
> chưa chạy huấn luyện Qwen2.5 hoặc benchmark T4 trong môi trường tạo notebook.
> Mọi bảng/biểu đồ bên dưới lấy từ kết quả thực khi bạn chạy, không có số liệu minh họa.


## 1. Cài cùng môi trường


In [ ]:
import base64, hashlib, io, os, subprocess, sys, zipfile
from pathlib import Path
SOURCE = Path("/content/t4decode_source")
blob = base64.b64decode("...")
assert hashlib.sha256(blob).hexdigest() == "768e62895610ee41f8f9dc08041c0af2f94a09ed42ca96a7cca6973eaf4a9d42", "Gói mã nguồn bị thay đổi"
SOURCE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for member in z.namelist():
        assert (SOURCE/member).resolve().is_relative_to(SOURCE.resolve())
    z.extractall(SOURCE)
requirements = [s.strip() for s in (SOURCE/"requirements-colab.txt").read_text().splitlines()
                if s.strip() and not s.lstrip().startswith("#")]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + requirements)
sys.path.insert(0, str(SOURCE))
os.environ["PYTHONPATH"] = str(SOURCE) + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Đã cài mã nguồn dùng chung:", SOURCE)


## 2. Dùng đúng PROJECT và policy đã khóa


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROFILE = "standard"  # "smoke" cho lần kiểm tra ngắn; dùng project khác khi đổi profile
PROJECT = Path("/content/drive/MyDrive") / f"T4Decode_{PROFILE}"
PROJECT.mkdir(parents=True, exist_ok=True)

from t4decode.engine import environment
from t4decode.core import read_json
import json
print(json.dumps(environment(), ensure_ascii=False, indent=2))

def run_step(command, *extra):
    subprocess.check_call([sys.executable, "-m", "t4decode.cli", command,
                           "--project", str(PROJECT), *extra])

print("Thư mục dùng chung cả ba notebook:", PROJECT)


In [ ]:
assert (PROJECT/"selected_policy.json").exists(), "Hoàn thành notebook 02 trước."
policy = read_json(PROJECT/"selected_policy.json")
print("Cấu hình khóa từ validation:", policy["candidate"])


## 3. Chạy test mới và đo lại các đối chứng

Mặc định chạy FP16, INT8, NF4, cấu hình được chọn và draft chưa train tương ứng nếu có.
Bật FULL_ABLATION để đo đủ tám cấu hình. Không cập nhật policy từ kết quả test.
Bộ test kỹ thuật giả lập không đại diện chất lượng tiếng Việt tổng quát; bổ sung benchmark thực
trước khi khẳng định tính tổng quát trong báo cáo cuối kỳ.


In [ ]:
FULL_ABLATION = False
run_step("evaluate", *(["--all-candidates"] if FULL_ABLATION else []))


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
run_dir = Path(read_json(PROJECT/"latest_test.json")["run"])
display(pd.read_csv(run_dir/"summary.csv"))
print("Dữ liệu thô:", run_dir)


In [ ]:
display(Markdown((run_dir/"REPORT.md").read_text()))


## 4. Biểu đồ từ kết quả đo thực

Throughput dưới đây bao gồm prefill + decoding, không phải tốc độ decode-only.


In [ ]:
import matplotlib.pyplot as plt
table = pd.read_csv(run_dir/"summary.csv").set_index("candidate")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
table["throughput_tok_s"].plot.bar(ax=axes[0], color="#167D8D")
axes[0].set(title="Throughput thực tế", ylabel="Output tokens/s (gồm prefill)", xlabel="")
table["peak_vram_gib"].plot.bar(ax=axes[1], color="#D88C34")
axes[1].axhline(policy["budget_gib"], color="red", linestyle="--", label="Ngân sách lựa chọn")
axes[1].set(title="Peak VRAM đo được", ylabel="GiB", xlabel=""); axes[1].legend()
plt.tight_layout(); plt.savefig(run_dir/"comparison.png", dpi=180, bbox_inches="tight"); plt.show()


## 5. Sử dụng hệ thống ngay trong Colab

Cell này nạp cấu hình đã chọn một lần. Giao diện không mở public tunnel.
Đo tổng thời gian generate và TTFT; văn bản hiển thị khi generate hoàn tất.
Đây là demo greedy có giới hạn 256 token đầu ra; không dùng thời gian UI để tính benchmark.
Nếu policy vượt giới hạn khi prompt mới dài hơn workload, ứng dụng báo lỗi để bạn điều chỉnh.


In [ ]:
from t4decode.engine import Engine
from IPython.display import clear_output
import ipywidgets as widgets
ui_engine = Engine(PROJECT, policy["candidate"])
ui_engine.warmup()
question = widgets.Textarea(value="Giải thích KV cache bằng một ví dụ ngắn.",
    layout=widgets.Layout(width="100%", height="100px"))
button = widgets.Button(description="Sinh câu trả lời", button_style="primary")
release = widgets.Button(description="Giải phóng GPU")
out = widgets.Output()

def answer(_):
    button.disabled = True
    with out:
        clear_output(wait=True)
        try:
            result = ui_engine.run(question.value, max_new_tokens=256)
            print(result["text"])
            print(f"\nGenerate: {result['total_s']:.2f}s | TTFT: {result['ttft_s']:.2f}s | "
                  f"VRAM: {result['peak_vram_gib']:.2f} GiB | {result['output_tokens']} token")
        except Exception as error:
            print(type(error).__name__, str(error))
    button.disabled = False

def free_gpu(_):
    ui_engine.close()
    button.disabled = True
    release.disabled = True
    with out:
        print("Đã giải phóng GPU. Chạy lại cell để mở demo mới.")

button.on_click(answer); release.on_click(free_gpu)
display(question, widgets.HBox([button, release]), out)


## 6. Xuất dữ liệu báo cáo

Chỉ tải kết quả/manifest/policy; checkpoint và model giữ trong PROJECT để tránh ZIP quá lớn.


In [ ]:
from google.colab import files
import zipfile
export = Path("/content/T4Decode_measured_results.zip")
with zipfile.ZipFile(export, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("manifest.json", "selected_policy.json"):
        z.write(PROJECT/name, name)
    for p in run_dir.rglob("*"):
        if p.is_file():
            z.write(p, str(Path("test_run")/p.relative_to(run_dir)))
files.download(str(export))
